# BTP step S1c, attempt 2 on Kaggle: SAE validation and E0 only (gemma-2-2b-it)

Attempt 1 (notebook16b0352a71 v1) generated and scored the 1,000 pre-registered traces, then hit CUDA error 719 on the first
SAE-validation forward pass (`reports/m0/s1c_attempt1_failure.txt`). This run reuses those traces (dataset `btp-s1c-traces`,
sha256 `2f7e58bf4024aeed225e9abe3b6c987f5fbefee9011643d8e2acb86c4d66e776`, see `reports/m0/m0_traces_provenance.txt`), runs a CUDA_LAUNCH_BLOCKING preflight, then steps 2-3 of
`scripts/s1_sae.sh` with the same arguments. No analysis code changes.
**Inputs:** `btp-code` (with `n_devices` in `analysis/sae_validation.py`), `btp-s1c-traces`; GPU T4 x2, Internet on, secret `HF_TOKEN`.

In [ ]:
!nvidia-smi
!python --version
import torch
print(torch.cuda.device_count(), [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())])
assert torch.cuda.device_count() >= 2, 'Need 2 GPUs: set Accelerator to GPU T4 x2'
!df -h /kaggle/working /root /tmp

In [ ]:
# Find the code under /kaggle/input: Kaggle may keep btp_code.zip or auto-extract it into a folder.
import glob, os, shutil, zipfile
DST = '/kaggle/working/Reasoning-Failure-Diagnosis'
shutil.rmtree(DST, ignore_errors=True)
zips = glob.glob('/kaggle/input/**/btp_code.zip', recursive=True)
reqs = [p for p in glob.glob('/kaggle/input/**/requirements.txt', recursive=True)
        if os.path.exists(os.path.join(os.path.dirname(p), 'scripts', 's1_equivalence.sh'))]
if zips:
    print('zip:', zips[0])
    tmp = '/kaggle/working/_unzip'
    shutil.rmtree(tmp, ignore_errors=True)
    zipfile.ZipFile(zips[0]).extractall(tmp)
    inner = [os.path.dirname(p) for p in glob.glob(tmp + '/**/requirements.txt', recursive=True)
             if os.path.exists(os.path.join(os.path.dirname(p), 'scripts', 's1_equivalence.sh'))]
    assert inner, 'no project root inside the zip'
    shutil.copytree(min(inner, key=len), DST)
    shutil.rmtree(tmp)
elif reqs:
    src = os.path.dirname(min(reqs, key=len))
    print('folder:', src)
    shutil.copytree(src, DST)
else:
    raise FileNotFoundError('btp_code not found under /kaggle/input; add the btp-code dataset as input')
assert 'numpy>=2.0' in open(f'{DST}/requirements.txt').read(), 'stale requirements.txt'
print(sorted(os.listdir(DST)))
assert 'hf_reference' in open(f'{DST}/harness/equivalence.py').read(), 'old btp-code version: update the dataset input'
assert 'n_devices' in open(f'{DST}/analysis/sae_validation.py').read(), 'old btp-code version (no S1c 2-GPU patch): update the dataset input'
assert 'Date committed:** 2026-11-11' in open(f'{DST}/prereg/E0.md').read(), 'E0 pre-registration not committed'

In [ ]:
# requirements.txt needs Python 3.13 (TransformerLens 2.x caps numpy<2 on <=3.12), so build a 3.13 venv with uv.
!pip -q install uv
!uv venv --seed --python 3.13 /kaggle/working/venv
!uv pip install --python /kaggle/working/venv/bin/python -r /kaggle/working/Reasoning-Failure-Diagnosis/requirements.txt
os.environ['PATH'] = '/kaggle/working/venv/bin:' + os.environ['PATH']
os.environ['VIRTUAL_ENV'] = '/kaggle/working/venv'
!which python && python -c "import torch, transformer_lens, numpy; print(torch.__version__, torch.cuda.is_available(), torch.cuda.device_count(), numpy.__version__)"

In [ ]:
from kaggle_secrets import UserSecretsClient
os.environ['HF_TOKEN'] = UserSecretsClient().get_secret('HF_TOKEN')   # never printed
print('HF_TOKEN set:', bool(os.environ['HF_TOKEN']))

In [ ]:
!cd /kaggle/working/Reasoning-Failure-Diagnosis && python -m pytest -q
assert _exit_code == 0, 'pytest failed'

In [ ]:
# 1. Traces from attempt 1, verified by hash
import hashlib
P = '/kaggle/working/Reasoning-Failure-Diagnosis'
srcs = glob.glob('/kaggle/input/**/m0_traces_scored.jsonl', recursive=True)
assert len(srcs) == 1, srcs
os.makedirs(P + '/data/generated/gemma-2-2b-it', exist_ok=True)
TR = P + '/data/generated/gemma-2-2b-it/m0_traces_scored.jsonl'
shutil.copy(srcs[0], TR)
h = hashlib.sha256(open(TR, 'rb').read()).hexdigest()
print(srcs[0], h, sum(1 for _ in open(TR)), 'lines')
assert h == '2f7e58bf4024aeed225e9abe3b6c987f5fbefee9011643d8e2acb86c4d66e776', 'trace file hash mismatch'

In [ ]:
# 2. GPU state before the run
!nvidia-smi

In [ ]:
# 3. Diagnostic preflight (CUDA_LAUNCH_BLOCKING=1): lengths, then TL loaded exactly as sae_validation.run does
PRE = r"""
import json, statistics, torch
from harness.config import model_config
from harness.generate import read_jsonl
from harness.models import load_hf, load_tl, load_tokenizer, pick_device
recs = read_jsonl('data/generated/gemma-2-2b-it/m0_traces_scored.jsonl')[:200]
lens = [len(r['prompt_ids']) + len(r['output_ids']) for r in recs]
print('first 200 records: min', min(lens), 'median', statistics.median(lens), 'max', max(lens), '| record 0:', lens[0])
cfg = model_config('gemma-2-2b-it')
device = pick_device(None)
tok = load_tokenizer(cfg)
hf = load_hf(cfg, device='cpu')
model = load_tl(cfg, hf_model=hf, tokenizer=tok, device=device, n_devices=2)
del hf
print('soft caps', model.cfg.output_logits_soft_cap, model.cfg.attn_scores_soft_cap)
print('block devices', sorted({str(next(b.parameters()).device) for b in model.blocks}), 'embed', model.embed.W_E.device, 'unembed', model.unembed.W_U.device)
for name, i in (('record 0', 0), ('longest', max(range(len(lens)), key=lens.__getitem__))):
    for d in range(torch.cuda.device_count()):
        torch.cuda.reset_peak_memory_stats(d)
    ids = recs[i]['prompt_ids'] + recs[i]['output_ids']
    with torch.no_grad():
        logits = model(torch.tensor([ids], device=model.cfg.device))
    torch.cuda.synchronize()
    print(name, 'index', i, 'tokens', len(ids), 'logits finite', bool(torch.isfinite(logits).all()),
          'max_memory_allocated MiB', [round(torch.cuda.max_memory_allocated(d) / 2**20) for d in range(torch.cuda.device_count())])
print('PREFLIGHT OK')
"""
open('/tmp/preflight.py', 'w').write(PRE)
!cd /kaggle/working/Reasoning-Failure-Diagnosis && CUDA_LAUNCH_BLOCKING=1 PYTHONPATH=/kaggle/working/Reasoning-Failure-Diagnosis python /tmp/preflight.py
assert _exit_code == 0, 'preflight failed (traceback above)'

In [ ]:
# 4. SAE validation (s1_sae.sh step 2, same arguments), CUDA_LAUNCH_BLOCKING=1
!cd /kaggle/working/Reasoning-Failure-Diagnosis && bash -c 'set -o pipefail; CUDA_LAUNCH_BLOCKING=1 python -m analysis.sae_validation --model gemma-2-2b-it --traces data/generated/gemma-2-2b-it/m0_traces_scored.jsonl --max-traces 200 --n-devices 2 --out reports/m0/sae_validation_gemma-2-2b-it.json 2>&1 | tee /kaggle/working/s1c_sae.log'
assert _exit_code == 0 and os.path.exists(P + '/reports/m0/sae_validation_gemma-2-2b-it.json'), 'SAE validation failed: see /kaggle/working/s1c_sae.log'

In [ ]:
# 5. E0 (s1_sae.sh step 3, same arguments)
!cd /kaggle/working/Reasoning-Failure-Diagnosis && bash -c 'set -o pipefail; (python -m analysis.sparsity_e0 collect --model gemma-2-2b-it --traces data/generated/gemma-2-2b-it/m0_traces_scored.jsonl --sae-report reports/m0/sae_validation_gemma-2-2b-it.json --max-traces 1000 --n-devices 2 --out data/generated/gemma-2-2b-it/e0_tokens.csv.gz && python -m analysis.sparsity_e0 analyse --tokens data/generated/gemma-2-2b-it/e0_tokens.csv.gz --n-layers 26 --out reports/m0/e0_gemma-2-2b-it.json) 2>&1 | tee /kaggle/working/s1c_e0.log'
assert _exit_code == 0 and os.path.exists(P + '/reports/m0/e0_gemma-2-2b-it.json'), 'E0 failed: see /kaggle/working/s1c_e0.log'

In [ ]:
# 6. Collect outputs and summarise
OUT = '/kaggle/working/out'
shutil.rmtree(OUT, ignore_errors=True); os.makedirs(OUT)
for f in glob.glob(P + '/reports/m0/*') + ['/kaggle/working/s1c_sae.log', '/kaggle/working/s1c_e0.log', P + '/data/generated/gemma-2-2b-it/e0_tokens.csv.gz']:
    shutil.copy(f, OUT)
for f in sorted(os.listdir(OUT)):
    print(os.path.getsize(f'{OUT}/{f}'), f, hashlib.sha256(open(f'{OUT}/{f}', 'rb').read()).hexdigest())
import json
v = json.load(open(OUT + '/sae_validation_gemma-2-2b-it.json'))
for r in v['layers']: print(f"layer {r['layer']:2d} FVU {r['fvu']:.4f} dCE {r['delta_ce']:.4f} tokens {r['n_tokens']} {'PASS' if r['pass'] else 'FAIL'}")
print('passing layers:', v['passing_layers'])
print(json.dumps(json.load(open(OUT + '/e0_gemma-2-2b-it.json')), indent=1))
!nvidia-smi